__Markus Mulvihill__

__Last updated April 2026__

# Necessary Packages and Modules

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import sys

In [2]:
sys.path.append(os.path.abspath(os.path.join('..', 'src')))
import ext_kalman_filter
import icing_true
import icing_model
import patient_cohort
import utils

# Introduction

Within this file, different methods of parameter estimation will be implemented and compared to predict the Insulin Sensitivity $\, SI$ in patients

# Setup
## Patient Cohort
+ 50 patients
+ Simulation over 60 hours
+ Initial $BG$ (mmol/L) is chosen from a log normal distribution of $\mu = 7.6$ and $\sigma=1.3$
+ Initial $Q$ and $I$ is 15 mu/L
+ Inital $P1$ and $P2$ is assumed to be 0 mmol/L
+ $\,SI$ for the patients will be set constant at $2.2 \cdot 10^{-4}$ 
+ $u_{ex}(t) = 75 \cdot e^{-\left(\frac{\log(2)}{300}\right) \cdot \left((t + 120) \mod (300)\right)}$ (mU/min)
+ $\,PN(t) = e^{-\left(\frac{\log(2)}{300}\right) \cdot \left(t \mod (300)\right)}$ (mmol/min)
+ $D(t) = 0.24$ (mmol/min)
+ True dynamics are derived from the ICING model with time steps $\,dt=1$ min
+ Blood Glucose measurements occur hour that contain Gaussian measurement noise $N(\mu = 0, \sigma^2 = 0.25^2)$

# Experiment
+ Every hour, the $\,SI$ parameter will be estimated from each patient
+ Parameter Estimation Methods
    1. State Augmentation of Extended Kalman Filter and Particle Filter Estimators
    2. MCMC with Gaussian filtering-based energy function
    3. MCMC with Rao–Blackwellized particle filter
+ The Mean Absolute Relative Difference (MARD) will be used as the error metric to compare the different methods 

In [20]:
BG_logmu = 7.6
BG_logsigma = 1.3
num_patients = 50
sim_hours = 60
dt = 1
t = np.arange(0, sim_hours*60+1, dt)
dtmeas = 60
t_meas = np.arange(0, sim_hours*60+1, dtmeas)
meas_noise_std = 0.25
SI_const = 2.2e-4
SI_init = 2.0e-4
PatientCohort = patient_cohort.PatientCohort(num_patients=num_patients, sim_hours=sim_hours, dt=dt, dtmeas=dtmeas, meas_noise_std=meas_noise_std, BG_params=[BG_logmu, BG_logsigma])
uex_func = utils.gen_uex_func(type="constant")
PN_func = utils.gen_PN_func()
D_func = utils.gen_D_func()
SI_func = utils.gen_SI_func(SI_const=SI_const)
SI_true = [SI_func(ts) for ts in t_meas]
input_functions = {"D": D_func, "PN": PN_func, "Uex": uex_func}
patient_data = PatientCohort.patient_data(uex_func=uex_func, PN_func=PN_func, D_func=D_func, SI_func=SI_func)

## State Augmentation

+ State Variables are $\dot{\,BG}$, $\dot{Q}$, $\dot{I}$, $\dot{\,P1}$, $\dot{\,P2}$, $u_{en}$, $\,SI$ 
+ The initial states for each patients are assigned as $\begin{bmatrix} \,BG \sim \,log N(7.6, 1.3) & 15 & 15 & 0 & 0 & k_1 e^{-\frac{k_2}{k_3}15} & 2.0 \cdot 10^{-4} \end{bmatrix}$
+ Process Noise is $\text{diag} \left( \begin{bmatrix} \frac{1e^{-4}}{120} & \frac{1e^{-4}}{120} & \frac{1e^{-4}}{120} & \frac{1e^{-4}}{120} & \frac{1e^{-4}}{120} & 0 & \frac{1e^{-12}}{120} \end{bmatrix} \right)$
+ Initial states for each particle (N=750) is sampled from a Gaussian distribution with a mean of the initial state and variance of the process noise

In [ ]:
SI_est_mard_EKF_augment = np.zeros((num_patients,))
SI_est_mard_PF_augment = np.zeros_like(SI_est_mard_EKF_augment)
num_states = 7
process_noise_vars =  ((1e-4)/120)*np.ones((num_states,))
process_noise_vars[-1] = (1e-12)/120
y0 = [BG_logmu, 15, 15, 0, 0]
initial_state = np.append(y0, [icing_true.ICINGTrue().params['k1']*np.exp(-y0[2]*icing_true.ICINGTrue().params['k2']/icing_true.ICINGTrue().params["k3"]), SI_init])
initial_state = np.expand_dims(initial_state, axis=1)
Model = icing_model.ICINGModel(params=None, dt=dt, initial_state=initial_state, process_noise_vars=process_noise_vars, measurement_noise_var=meas_noise_std**2, u_funcs=input_functions)
EKF = ext_kalman_filter.ExtendedKalmanFilter(num_states=num_states, model=Model, ts_meas=dtmeas, state_augment=True)

for i in range(num_patients):
    G_meas = patient_data[i]['BG_meas']
    sim = EKF.simulate(t=t, t_meas=t_meas, G_meas=G_meas, filter=EKF)
    saved_state, saved_noise_var, G_est, Q_est, I_est, P1_est, P2_est, Uen_est, SI_est, G_pred  = sim.run()
    # SI_est_mard_EKF_augment[i] = np.mean(np.abs((SI_est - SI_true)*100/SI_true))